In [1]:
import pandas as pd
import ast
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import normalize

In [2]:
df = pd.read_csv('../data/tmdb_movies_ru.csv')
df = df.dropna(subset=['poster_path'])
df = df[df['poster_path'].astype(str).str.strip() != '']

In [3]:
def parse_genres(genre_string):
    try:
        return ast.literal_eval(genre_string)
    except:
        return []

df['genres_list'] = df['genres'].apply(parse_genres)

mlb = MultiLabelBinarizer()
genre_ohe = mlb.fit_transform(df['genres_list'])

genre_ohe_normalized = normalize(genre_ohe, norm='l2')
print(f"Извлечено уникальных жанров: {genre_ohe.shape[1]}")

Извлечено уникальных жанров: 19


In [4]:
model = SentenceTransformer('intfloat/multilingual-e5-small')

texts_to_embed = []
for _, row in df.iterrows():
    title = str(row['title']) if pd.notna(row['title']) else ""
    genres = str(row['genres']) if pd.notna(row['genres']) else ""
    overview = str(row['overview']) if pd.notna(row['overview']) else ""
    
    combined_text = f"passage: Название: {title}. Жанры: {genres}. Описание: {overview}"
    texts_to_embed.append(combined_text)

print("Генерация текстовых эмбеддингов E5 (это займет некоторое время)...")
text_embeddings = model.encode(
    texts_to_embed, 
    batch_size=64, 
    show_progress_bar=True, 
    normalize_embeddings=True # Текстовые векторы сразу нормализуются
)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Генерация текстовых эмбеддингов E5 (это займет некоторое время)...


Batches:   0%|          | 0/280 [00:00<?, ?it/s]

In [5]:
GENRE_WEIGHT = 1.5 

weighted_genres = genre_ohe_normalized * GENRE_WEIGHT

hybrid_embeddings = np.hstack((text_embeddings, weighted_genres))

final_embeddings = normalize(hybrid_embeddings, norm='l2').astype(np.float32)

print(f"Размерность текстового вектора: {text_embeddings.shape[1]}")
print(f"Размерность жанрового вектора: {genre_ohe.shape[1]}")
print(f"Итоговая размерность гибридного вектора: {final_embeddings.shape[1]}")

Размерность текстового вектора: 384
Размерность жанрового вектора: 19
Итоговая размерность гибридного вектора: 403


In [ ]:
bin_path = '../data/movie_embeddings.bin'

with open(bin_path, 'wb') as f:
    np.array(final_embeddings.shape, dtype=np.uint32).tofile(f)
    final_embeddings.tofile(f)

df[['id', 'title']].to_csv('../data/movie_mapping.csv', index=False)
movie_ids = df['id'].values.astype(np.uint32)
movie_ids.tofile('../data/movie_ids.bin')

print("Гибридные эмбеддинги и маппинг успешно сохранены!")

Гибридные эмбеддинги и маппинг успешно сохранены!
